* 重新 review GPT 的过程

    * input_ids: 1*1024, 一个 (bs) 长度为 1024 的 token ids
    * last_hidden_states: 1*1024*768
        * last layer hidden states of (transformer)
        * (casual) self-attention + ffn
    * lm_logits: 1*1024*50257
        * lm head, 将每一个位置上的 token 的 hidden state, 映射到整个词表维度上的概率分布输出

* shift labels 与损失计算

```python
labels = labels.to(lm_logits.device)

# Shift so that tokens < n predict n
shift_logits = lm_logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()

# Flatten the tokens
loss_fct = CrossEntropyLoss()
loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

### 2. 详细易懂的解释

前两张图讲的是“理论公式”，这张图则是带我们进入了**“实战代码”**。这张幻灯片展示了在使用 PyTorch 框架训练 GPT 模型时，数据在网络中是怎样流动的，以及最后那一步极其关键的“错位计算 Loss”是怎么用代码写出来的。

我们可以把整个过程分成两半来看：上半部分是**数据的变形记（前向传播）**，下半部分是**对答案算分（计算损失）**。

#### 上半部分：数据在 GPT 肚子里的变形记
幻灯片列出了三个关键的张量（矩阵）的形状变化：

1.  **`input_ids`: `1 * 1024`**
    *   这是喂给模型的**输入数据**。
    *   `1` 代表 Batch Size (批次大小)，也就是一次处理 1 句话。
    *   `1024` 代表这句话有 1024 个字（Token）。
    *   **通俗理解**：你扔给模型一本 1024 个字的小册子。

2.  **`last_hidden_states`: `1 * 1024 * 768`**
    *   输入数据经过了 GPT 模型深厚的内部神经网络（也就是图上写的 Transformer 里的 self-attention 和 ffn）。
    *   注意图中的 `casual` 是个常见的拼写错误，正确的应该是 **`causal` (因果的)**，意思是模型只能看前面的词，不能偷看后面的词（这叫因果掩码注意力）。
    *   `768` 是隐藏层维度。模型把那 1024 个字，每一个字都变成了长度为 768 的数字向量，用来深刻理解这个字在上下文里的含义。

3.  **`lm_logits`: `1 * 1024 * 50257`**
    *   这是模型出来的**原始预测结果**。
    *   数据经过最后的 `LM head`（语言模型头）。
    *   `50257` 是**词表大小**（这正是 GPT-2 的经典词表大小）。它把每个字那 768 维的深奥理解，展开成了 50257 个分数。
    *   **通俗理解**：对于这 1024 个位置上的每一个字，模型都给出了一个涵盖 50257 个选项的“下一个字预测分数表”。

#### 下半部分：极其巧妙的 `shift labels` (错位对答案)
大语言模型的任务是**“根据当前字，预测下一个字”**。这在代码里是怎么实现的呢？这就是这段代码最核心的逻辑：**错位（Shift）**。

举个极简的例子，假设我们的句子是 `A B C D`。
*   模型在位置 `A` 产生了一个预测结果，它应该去和 `B` 对答案。
*   模型在位置 `B` 产生了一个预测结果，它应该去和 `C` 对答案。
*   模型在位置 `C` 产生了一个预测结果，它应该去和 `D` 对答案。
*   模型在位置 `D` 产生的预测结果，没有东西可以对答案了（因为句子结束了）。

**代码是怎么做的？**
*   **`shift_logits = lm_logits[..., :-1, :]`**
    这句话的意思是：把模型所有的预测结果拿出来，**除了最后一个**。我们保留位置 `A`, `B`, `C` 的预测结果。（扔掉了对 `D` 后面的预测，因为没标准答案）。
*   **`shift_labels = labels[..., 1:]`**
    这句话的意思是：把标准答案拿出来，**除了第一个**。我们保留真实的词 `B`, `C`, `D`。（因为 `A` 前面没有字，不是任何字的“下一个字”）。

**对齐了！**
现在，`shift_logits` (基于 `A, B, C` 做的预测) 和 `shift_labels` (真实的下一个字 `B, C, D`) 在长度上完全对齐了。

#### 最后一步：算总分
```python
loss_fct = CrossEntropyLoss()
loss = loss_fct(shift_logits.view(-1, ...), shift_labels.view(-1))

* BERT: 双向注意力 (bidirectional self attention)

  $$ \text{Attention}(Q^{(n \times d_k)}, K^{(n \times d_k)}, V^{(n \times d_v)}) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

* GPT: 单向因果注意力 (causal self attention)

  $$ \text{Attention}(Q^{(n \times d_k)}, K^{(n \times d_k)}, V^{(n \times d_v)}) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V $$

  * $M_{ij} = 0, j > i$ *(注：原图符号略模糊，结合矩阵推断此处应为大于号)*
  * $M_{ij} = 1, j \leq i$

  $$ M = \begin{pmatrix}
  1 & -\infty & -\infty & \cdots & -\infty \\
  1 & 1 & -\infty & \cdots & -\infty \\
  1 & 1 & 1 & \cdots & -\infty \\
  \vdots & \vdots & \vdots & \ddots & \vdots \\
  1 & 1 & 1 & \cdots & 1
  \end{pmatrix}_{n \times n} $$

我们来看看这个由 $1$ 和 $-\infty$（负无穷）组成的三角形矩阵：行与列：假设矩阵的行（$i$）代表“当前正在看的词”，列（$j$）代表“句子里的其他词”。对角线及左下角（都是 1）：代表 $j \leq i$，也就是过去和现在的词。当前词可以看到它们，加上 1 对后续计算概率的相对大小没有本质影响。右上角（都是 $-\infty$）：代表 $j > i$，也就是未来的词。为什么要用 $-\infty$（负无穷）？这是这页 PPT 里最巧妙的数学设计。注意公式里外面套了一个 softmax 函数。softmax 的核心操作是对自然底数求指数，即 $e^x$。当我们把 $-\infty$ 强行加到“未来词”的得分上时，由于 $e^{-\infty} = 0$。这就使得在最终算出来的概率分布中，所有未来词的被关注概率全部变成了绝对的 0。

* T5: encoder 输出 K/V (取值相同), decoder 输出 Q, 两者做 Cross attention

  $$ \text{Encoder Self-Attention} : \quad \text{Attention}(Q^{(n \times d_k)}, K^{(n \times d_k)}, V^{(n \times d_v)}) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

  $$ \text{Decoder Masked Self-Attention} : \quad \text{Attention}(Q^{(m \times d_k)}, K^{(m \times d_k)}, V^{(m \times d_v)}) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V $$

  $$ \text{Cross-Attention} : \quad \text{Attention}(Q^{(m \times d_k)}, K^{(n \times d_k)}, V^{(n \times d_v)}) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

## CrossEntropyLoss

计算的角度

* labels 起到选择的作用
* ignore_index: 过滤 (-100)
    * PPL 计算的时候会用到

```python
# -100, labels (token id) 提供一个选择器

# Example of target with class indices
loss = nn.CrossEntropyLoss()

#### 概念 1：为什么说“labels 起到选择的作用”？
回到我们在第二张图讲过的多分类交叉熵：系统在算分时，**根本不关心模型给错误答案打了多少分，只盯着正确答案的分数**。

在代码底层，为了加速计算，程序不会傻傻地把所有成千上万个词表的概率都拿去算一遍公式。
* **假设词表大小是 50000**。
* **模型输出 (`logits`)**：得到了一个长度为 50000 的概率数组。
* **真实标签 (`labels`)**：告诉你正确答案的词汇编号是 `886`。
* **底层操作**：`CrossEntropyLoss` 根本不看另外 49999 个分数。它直接用 `886` 作为一个**“索引（选择器）”**，跑到模型的输出数组里，把第 `886` 个位置上的分数揪出来，然后仅仅对这一个分数求 `-log(P)`。

这就叫“labels 起到选择的作用”，它极大节省了计算机的算力。

#### 概念 2：什么是 `ignore_index = -100`？（重点）
在做大模型训练时，我们每次喂给模型的数据通常是一个“批次（Batch）”，比如一次放 4 句话进去。
但问题来了：**这 4 句话的长度肯定不一样啊！**
* 句子 A：我 爱 吃 苹 果。（5个词）
* 句子 B：天 气 真 好。（4个词）

为了让它们能对齐成一个规整的矩阵（矩形），程序员会在短句子后面垫入“无意义的填充词（Padding）”：
* 句子 A：我 爱 吃 苹 果
* 句子 B：天 气 真 好 **[PAD]**

**计算 Loss 时遇到 [PAD] 怎么办？**
模型在预测到最后一步时，可能会乱猜各种词。但这个位置本来就是强行补上的空白，**我们不应该因为模型猜错了这个空白位置而扣它的分**。
* 在 PyTorch 中，默认规定了一个神奇的数字：**`-100`**。
* 我们把所有 `[PAD]` 对应的真实标签（label）设置为 `-100`。
* 当 `CrossEntropyLoss` 扫过数据，看到标签是 `-100` 时，它就会说：“哦，这是用来凑数的，我**忽略（ignore）**它，不把这个位置算进总 Loss 里。”

#### 概念 3：为什么“PPL 计算的时候会用到”？
我们前面讲过，PPL（困惑度）是由 Loss 换算过来的。
如果我们在算 Loss 的时候没有过滤掉那些用来凑数的 `[PAD]`（也就是没有忽略 -100），那么这些空白位置就会严重干扰总体分数的平均值。
过滤掉 -100 后，我们算出来的 PPL 才是**模型对真正的人类语言（有意义的词汇）的困惑程度**，这个指标才真实有效。

#### 概念 4：代码模拟部分
最下面的代码 `input = torch.randn(3, 5, requires_grad=True)` 是在用随机数模拟大模型的输出。
* `3`：可以理解为有 3 个字（Token）。
* `5`：可以理解为词表总共只有 5 个词。
* 这行代码生成了一个 $3 \times 5$ 的矩阵，用来在后续步骤中演示刚才讲的“选择器”和“-100 过滤”是如何在代码中实际生效的。

## Training & Inference/Generate

* llama2/3 inference code: autoregressive, token by token generation
    * https://github.com/meta-llama/llama3/blob/main/llama/generation.py#L179-L192C13
    * 天然隐式地存在一个mask matrix
    * 第一个单词，预测第二个单词，
    * 第一个单词+第二个单词 => 预测第三个单词
    * ...
* training 的时候，因为有 casual mask（下三角矩阵的存在），等价于 autoregressive, token by token
    * 显式地加 mask matrix，不让模型看到后边的结果
* 计算 PPL（语言模型训练好坏的一个指标）的过程就是已有文本的测试集，可以用 casual mask的方式实现自注意力，实现 autoregressive, token by token

概念 1：生成/推理阶段 (Inference/Generate) —— “天然隐式的 Mask”当我们真正在使用 Llama 2/3 或者 ChatGPT 时（也就是代码里的 inference code），模型确实是老老实实、一步一步来的：看到“第1个字”，预测出“第2个字”。把“第1个字 + 第2个字”喂给模型，预测出“第3个字”。循环往复...这叫 token by token generation。为什么说这里“天然隐式地存在一个 mask matrix（掩码矩阵）”？因为在生成的时候，未来还没发生！你想让它偷看第 4 个字，它也看不了，因为第 4 个字根本还没生成出来。现实世界的物理时间规则，自动替模型挡住了未来的信息。所以在这个阶段，程序员不需要在代码里强行加上我们在前面看到的那个 $-\infty$ 的掩码矩阵。

概念 2：训练阶段 (Training) —— “显式的 Causal Mask (下三角矩阵)”到了训练阶段，情况完全变了。训练时，我们手里是有完整的一句话（比如：“我爱吃苹果”）的标准答案的。为了追求 GPU 的极速并行计算，我们不可能让它一个字一个字慢慢学，而是把整句话一次性全部塞进模型里。但是，一次性全塞进去就会有个致命问题：模型会作弊！ 当它在预测“爱”的时候，它的眼睛可以同时瞟到后面标准答案里的“吃苹果”。怎么解决？这就用到了前面提过的 Causal Mask（因果掩码）。PPT 里提到的“下三角矩阵”，就是指那个左下角全是 1，右上角全是 $-\infty$ 的矩阵。我们“显式地（刻意在代码里）”加上这个矩阵。当它算“我”的时候，把后面的词遮住。当它算“爱”的时候，只能看到“我、爱”，把后面的词遮住。结论：在训练时，虽然我们是一次性把数据丢进去的，但由于有了这个巧妙的“下三角矩阵”遮挡，它在数学计算上，完美等价于（等价于 autoregressive）一个字一个字生成的环境。既防止了作弊，又实现了光速的并行计算。

概念 3：计算 PPL (测试集的困惑度评估)在训练结束或者进行中，我们要拿一些没见过的测试集来考一考模型，算一算它的 PPL（困惑度，越低越好）。考它的过程，本质上和“训练”是一样的：我们手里也有完整的测试集文本，为了算分算得快，我们同样不会让它一个字一个字去生成，而是直接采用加 Causal Mask 的方式，一次性得出整句话所有位置的预测概率，然后套用前面学过的 $PPL = e^{Loss}$ 公式，瞬间算出这批测试题的得分。总结：真正用它时（Inference）：现实时间充当了掩码，老老实实一个字一个字往外蹦。训练和考试时（Training / PPL）：为了高效，一次性看全篇，但用“数学矩阵（Causal Mask）”强行给它戴上眼罩，模拟出不能偷看未来的真实环境。